# NB07：生產級 RAG 優化與 Agentic RAG

**目標：** 理解 RAG 系統從「Demo」到「生產環境」的工程差距，並掌握 Agentic RAG 的核心概念與實作。

**學習成果：**
- 實作語意快取（Semantic Cache）節省 LLM 成本
- 掌握 ChromaDB Metadata 過濾提升搜索精度
- 用 asyncio 實現非同步並行檢索加速
- 計算並優化 RAG 系統的 API 成本
- 理解並實作 Self-RAG 自主檢索決策
- 理解並實作 CRAG 修正式檢索
- 掌握完整生產級 RAG 架構全貌

## 開場：Demo 和生產環境的差距

### 為什麼 RAG Demo 能跑，但生產環境很痛苦？

```
Demo 環境                        生產環境
────────────────                 ────────────────────────────
100 份文件                       10,000,000 份文件
1 個使用者                       10,000 個並發使用者
不在乎速度                       P95 延遲需 < 3 秒
不在乎成本                       每天 $500+ 的 API 費用
查詢品質不穩定無所謂             精準度、召回率需持續監控
```

### 本 Notebook 涵蓋的生產優化主題

| 主題 | 問題 | 解法 |
|------|------|------|
| **語意快取** | 重複查詢浪費 LLM 費用 | 相似查詢直接回傳快取答案 |
| **Metadata 過濾** | 向量搜索範圍太廣 | 先過濾再搜索，提升精準度 |
| **非同步並行** | 多路搜索依序等待太慢 | asyncio 並行執行 |
| **成本優化** | 不知道錢花在哪 | Token 計算 + 策略性降本 |
| **Self-RAG** | LLM 盲目回答，不思考是否需要檢索 | 自主決策何時要、不要檢索 |
| **CRAG** | 檢索品質差時仍強行生成 | 評估品質 + 自動修正策略 |

## 環境設定

In [ ]:
import os
import json
import time
import asyncio
import random
import math
from typing import Optional
from dotenv import load_dotenv
from openai import OpenAI
import chromadb
import tiktoken
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 載入 .env 中的 API Key
load_dotenv()

# 初始化 OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 初始化 ChromaDB（in-memory）
chroma_client = chromadb.Client()

# 設定中文字體
chinese_fonts = [f.name for f in fm.fontManager.ttflist
                 if any(kw in f.name for kw in ["Heiti", "PingFang", "Noto", "CJK", "Source Han"])]
if chinese_fonts:
    plt.rcParams["font.family"] = chinese_fonts[0]
else:
    plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False

print("✅ 環境設定完成")
print(f"OpenAI API Key 存在: {bool(os.getenv('OPENAI_API_KEY'))}")

## 基礎工具函式（後續所有部分共用）

In [ ]:
def create_embedding(text: str) -> list:
    """使用 OpenAI text-embedding-3-small 生成向量"""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding


def cosine_similarity(vec_a: list, vec_b: list) -> float:
    """計算兩向量的餘弦相似度"""
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))
    norm_b = math.sqrt(sum(b * b for b in vec_b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


def llm_call(prompt: str, system: str = "你是一個知識問答助理，請用繁體中文回答。",
             temperature: float = 0.1) -> str:
    """封裝 LLM 呼叫"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt}
        ],
        temperature=temperature
    )
    return response.choices[0].message.content.strip()


print("✅ 基礎工具函式已定義")

## 示範知識庫文件

建立帶有豐富 Metadata 的文件集，供後續各部分使用。

In [ ]:
# 帶 Metadata 的示範文件（10 份）
documents_with_metadata = [
    {"id": "d01", "content": "BERT 的雙向 Transformer 架構讓它能同時理解前後文，使用 MLM 和 NSP 預訓練任務。在 NLP 下游任務上表現優異。",
     "metadata": {"category": "LLM", "year": 2018, "source": "research_paper", "lang": "zh"}},
    {"id": "d02", "content": "GPT-4 是 OpenAI 2023 年發布的多模態模型，在律師考試達到前 10%，支援 128K context window。",
     "metadata": {"category": "LLM", "year": 2023, "source": "blog", "lang": "zh"}},
    {"id": "d03", "content": "ChromaDB 是輕量級向量資料庫，支援 Python、JavaScript，適合原型開發和小規模部署。提供 in-memory 和持久化兩種模式。",
     "metadata": {"category": "vector_db", "year": 2023, "source": "documentation", "lang": "zh"}},
    {"id": "d04", "content": "Pinecone 是雲端托管的向量資料庫服務，支援大規模生產部署，提供自動擴展和多租戶隔離。定價按向量數量計算。",
     "metadata": {"category": "vector_db", "year": 2023, "source": "documentation", "lang": "zh"}},
    {"id": "d05", "content": "RAG 系統的 P95 延遲目標通常設定在 3 秒以內。延遲分佈：向量搜索 30-100ms，Reranking 100-500ms，LLM 生成 500-3000ms。",
     "metadata": {"category": "production", "year": 2024, "source": "engineering_blog", "lang": "zh"}},
    {"id": "d06", "content": "語意快取（Semantic Cache）通過向量相似度識別重複查詢，避免重複呼叫 LLM。相似度閾值通常設為 0.95，可節省 30-60% 的 LLM 費用。",
     "metadata": {"category": "production", "year": 2024, "source": "engineering_blog", "lang": "zh"}},
    {"id": "d07", "content": "HNSW 索引適合高速近似搜索，建立慢但查詢快。IVF 索引適合大規模資料，需要訓練期。Flat 索引精確但不可擴展。",
     "metadata": {"category": "vector_db", "year": 2023, "source": "research_paper", "lang": "zh"}},
    {"id": "d08", "content": "Agentic RAG 讓 LLM 自主決定何時檢索、如何修正查詢、是否信任檢索結果。代表性工作包括 Self-RAG、CRAG、ReAct 框架。",
     "metadata": {"category": "LLM", "year": 2024, "source": "research_paper", "lang": "zh"}},
    {"id": "d09", "content": "成本優化：gpt-4o-mini 比 gpt-4o 便宜 30x，適合簡單查詢。Contextual compression 可減少 50-70% context token。批量 embedding 可降低 API 呼叫次數。",
     "metadata": {"category": "production", "year": 2024, "source": "engineering_blog", "lang": "zh"}},
    {"id": "d10", "content": "Multi-modal RAG 支援圖片、表格、PDF 的理解和檢索。GPT-4V 和 Gemini 支援圖片輸入，可以理解截圖中的文字和圖表。",
     "metadata": {"category": "LLM", "year": 2024, "source": "blog", "lang": "zh"}},
]

print(f"示範文件共 {len(documents_with_metadata)} 份：")
for doc in documents_with_metadata:
    m = doc["metadata"]
    print(f"  [{doc['id']}] category={m['category']}, year={m['year']}, source={m['source']}")

---
## Part 1：語意快取（Semantic Cache）

### 核心問題：重複查詢浪費大量成本

在生產環境中，使用者常常問非常相似的問題：
- 「什麼是 RAG？」
- 「請問 RAG 是什麼意思？」
- 「RAG 是什麼技術？」

這三個問題語意幾乎相同，但傳統快取（完全比對字串）視為三個不同查詢，每次都呼叫 LLM。

### 語意快取的運作原理

```
新查詢
  ↓
語意快取查詢（向量相似度）
  ↓
相似度 > 0.95？
  ├── YES → 直接回傳快取答案（省去 LLM 呼叫！）
  └── NO  → 完整 RAG pipeline → 存入快取
```

### 為什麼用向量相似度而不是字串匹配？

| 方法 | 「RAG 是什麼？」vs「請問什麼是 RAG」 | 問題 |
|------|--------------------------------------|------|
| 精確字串匹配 | 不同 → 不命中快取 | 遺漏大量語意相同查詢 |
| 語意向量相似度 | 相似度 ~0.97 → 命中快取 | 正確識別語意重複 |

In [ ]:
class SemanticCache:
    """
    語意快取：用向量相似度判斷查詢是否已有近似答案
    cache 中每筆記錄：(embedding, query, answer, timestamp)
    """
    def __init__(self, similarity_threshold: float = 0.95, max_size: int = 100):
        self.cache = []  # list of (embedding, query, answer, timestamp)
        self.threshold = similarity_threshold
        self.max_size = max_size
        self.hits = 0
        self.misses = 0

    def get(self, query: str):
        """若找到相似查詢則回傳 (answer, similarity)，否則回傳 (None, 0)"""
        if not self.cache:
            self.misses += 1
            return None, 0.0

        query_emb = create_embedding(query)
        best_sim = 0.0
        best_answer = None
        best_query = None

        for cached_emb, cached_query, cached_answer, _ in self.cache:
            sim = cosine_similarity(query_emb, cached_emb)
            if sim > best_sim:
                best_sim = sim
                best_answer = cached_answer
                best_query = cached_query

        if best_sim >= self.threshold:
            self.hits += 1
            print(f"  [快取命中] 相似度={best_sim:.4f} | 原始查詢: '{best_query[:40]}...'")
            return best_answer, best_sim

        self.misses += 1
        return None, best_sim

    def set(self, query: str, answer: str):
        """將查詢-答案對存入快取"""
        query_emb = create_embedding(query)
        self.cache.append((query_emb, query, answer, time.time()))
        # 超過上限時移除最舊的（LRU-lite）
        if len(self.cache) > self.max_size:
            self.cache.pop(0)

    @property
    def hit_rate(self) -> float:
        total = self.hits + self.misses
        return self.hits / total if total > 0 else 0.0

    def stats(self) -> dict:
        return {
            "size": len(self.cache),
            "hits": self.hits,
            "misses": self.misses,
            "hit_rate": round(self.hit_rate, 4),
            "threshold": self.threshold
        }


print("✅ SemanticCache 類別已定義")

In [ ]:
# ── 建立快速的 RAG 生成函式（供快取 Demo 使用）──
def simple_rag_generate(query: str) -> str:
    """簡化版 RAG 生成（直接用 LLM 回答，模擬完整 pipeline 的最後一步）"""
    return llm_call(
        f"請用 2-3 句話回答以下問題（繁體中文）：{query}",
        temperature=0.1
    )


# ── 建立語意快取並模擬 20 個查詢 ──
cache = SemanticCache(similarity_threshold=0.95, max_size=50)

# 模擬查詢集（包含原始查詢 + 近義重複）
simulated_queries = [
    # 第 1 組：關於 RAG
    "什麼是 RAG 技術？",
    "請問 RAG 是什麼意思？",           # 近義重複
    "RAG 是什麼技術，有什麼用途？",    # 近義重複
    # 第 2 組：關於向量資料庫
    "ChromaDB 和 Pinecone 有什麼差別？",
    "ChromaDB 跟 Pinecone 的比較",      # 近義重複
    # 第 3 組：關於成本優化
    "如何降低 RAG 系統的 API 費用？",
    "RAG 成本優化有哪些方法？",         # 近義重複
    "怎麼節省 LLM 的使用費用？",        # 近義重複
    # 第 4 組：關於延遲
    "RAG 系統的延遲來源是什麼？",
    "向量搜索需要多少時間？",
    "RAG pipeline 為什麼慢？",           # 近義重複
    # 第 5 組：關於 HNSW
    "HNSW 索引的原理是什麼？",
    "HNSW 是如何運作的？",              # 近義重複
    # 第 6 組：混合查詢（不同主題）
    "Self-RAG 和傳統 RAG 有何不同？",
    "什麼是 Agentic RAG？",
    "CRAG 的全名是什麼？",
    "語意快取的閾值怎麼設定？",
    "metadata 過濾如何提升搜索精度？",
    "gpt-4o-mini 比 gpt-4o 便宜多少？",
    "如何評估 RAG 系統的好壞？",
]

print(f"模擬 {len(simulated_queries)} 個查詢（含重複）")
print("=" * 60)

llm_call_count = 0
hit_rates_over_time = []

for i, query in enumerate(simulated_queries, 1):
    print(f"\n[{i:02d}] 查詢: {query}")

    # 先查快取
    cached_answer, sim = cache.get(query)

    if cached_answer is not None:
        print(f"  → 快取命中！節省一次 LLM 呼叫")
    else:
        # 快取未命中 → 呼叫 LLM
        answer = simple_rag_generate(query)
        cache.set(query, answer)
        llm_call_count += 1
        print(f"  → LLM 呼叫（第 {llm_call_count} 次），已存入快取")

    hit_rates_over_time.append(cache.hit_rate)

print("\n" + "=" * 60)
print("最終統計：")
stats = cache.stats()
for k, v in stats.items():
    print(f"  {k}: {v}")
print(f"\n  實際 LLM 呼叫次數: {llm_call_count} / {len(simulated_queries)} 次")
print(f"  節省呼叫: {len(simulated_queries) - llm_call_count} 次（{(1 - llm_call_count/len(simulated_queries))*100:.1f}%）")

In [ ]:
# ── 視覺化：快取命中率隨時間增長 ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左圖：快取命中率趨勢
ax1 = axes[0]
ax1.plot(range(1, len(hit_rates_over_time) + 1), hit_rates_over_time,
         marker="o", color="#2196F3", linewidth=2, markersize=5)
ax1.axhline(y=0.3, color="gray", linestyle="--", alpha=0.6, label="30% 基準線")
ax1.fill_between(range(1, len(hit_rates_over_time) + 1), hit_rates_over_time,
                 alpha=0.15, color="#2196F3")
ax1.set_title("語意快取命中率隨查詢數增長", fontsize=13)
ax1.set_xlabel("累計查詢數")
ax1.set_ylabel("快取命中率")
ax1.set_ylim(0, 1)
ax1.legend()
ax1.grid(alpha=0.3)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# 右圖：成本節省比較（長條圖）
ax2 = axes[1]
labels = ["無快取\n(全部 LLM 呼叫)", "語意快取\n(命中不呼叫 LLM)"]
no_cache_calls = len(simulated_queries)
with_cache_calls = llm_call_count
values = [no_cache_calls, with_cache_calls]
colors = ["#EF5350", "#4CAF50"]

bars = ax2.bar(labels, values, color=colors, width=0.4, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, values):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
             str(val), ha="center", fontsize=13, fontweight="bold")

ax2.set_title("LLM 呼叫次數比較", fontsize=13)
ax2.set_ylabel("LLM 呼叫次數")
ax2.set_ylim(0, no_cache_calls + 3)
ax2.grid(axis="y", alpha=0.3)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

saving_pct = (1 - with_cache_calls / no_cache_calls) * 100
ax2.annotate(f"節省 {saving_pct:.0f}%",
             xy=(1, with_cache_calls), xytext=(1.3, (no_cache_calls + with_cache_calls) / 2),
             fontsize=12, color="#4CAF50", fontweight="bold",
             arrowprops=dict(arrowstyle="->", color="#4CAF50"))

plt.suptitle("語意快取效果分析", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print("圖表已生成")

---
## Part 2：Metadata 過濾（Metadata Filtering）

### 核心問題：向量搜索範圍太廣

沒有過濾時，每次查詢都要計算和全量文件的相似度：

```
全量搜索 (10,000 docs):
  query → search all 10,000 vectors → top 5
  （結果可能包含過時文件、不相關分類）

Metadata 過濾 (10,000 docs，分類為 500 個 AI 文件):
  query + where:{"category": "LLM", "year": {"$gte": 2023}}
  → search only 500 vectors → top 5（更精準！成本更低！）
```

### ChromaDB Where 子句運算子

| 運算子 | 說明 | 範例 |
|--------|------|------|
| `$eq` | 等於 | `{"category": {"$eq": "LLM"}}` 或直接 `{"category": "LLM"}` |
| `$ne` | 不等於 | `{"lang": {"$ne": "en"}}` |
| `$gt` / `$gte` | 大於 / 大於等於 | `{"year": {"$gte": 2023}}` |
| `$lt` / `$lte` | 小於 / 小於等於 | `{"year": {"$lt": 2024}}` |
| `$in` | 包含在列表中 | `{"source": {"$in": ["blog", "research_paper"]}}` |
| `$and` | 且 | `{"$and": [{"category": "LLM"}, {"year": {"$gte": 2023}}]}` |
| `$or` | 或 | `{"$or": [{"category": "LLM"}, {"category": "vector_db"}]}` |

In [ ]:
# ── 建立帶 Metadata 的 ChromaDB collection ──
try:
    chroma_client.delete_collection("metadata_demo")
except Exception:
    pass

meta_collection = chroma_client.create_collection(
    name="metadata_demo",
    metadata={"hnsw:space": "cosine"}
)

print("建立向量索引（含 metadata）...")
for doc in documents_with_metadata:
    embedding = create_embedding(doc["content"])
    meta_collection.add(
        ids=[doc["id"]],
        embeddings=[embedding],
        documents=[doc["content"]],
        metadatas=[doc["metadata"]]
    )
    print(f"  ✅ [{doc['id']}] {doc['content'][:45]}...")

print(f"\n向量資料庫共 {meta_collection.count()} 份文件")

In [ ]:
def filtered_search(query: str, category: str = None, year_gte: int = None,
                    source: str = None, n_results: int = 3) -> dict:
    """帶 Metadata 過濾的向量搜索"""
    # 動態建立 where 子句
    conditions = []
    if category:
        conditions.append({"category": {"$eq": category}})
    if year_gte:
        conditions.append({"year": {"$gte": year_gte}})
    if source:
        conditions.append({"source": {"$eq": source}})

    if len(conditions) == 0:
        where_clause = None
    elif len(conditions) == 1:
        where_clause = conditions[0]
    else:
        where_clause = {"$and": conditions}

    query_emb = create_embedding(query)

    results = meta_collection.query(
        query_embeddings=[query_emb],
        n_results=min(n_results, meta_collection.count()),
        where=where_clause,
        include=["documents", "metadatas", "distances"]
    )

    return {
        "where_clause": where_clause,
        "ids": results["ids"][0],
        "documents": results["documents"][0],
        "metadatas": results["metadatas"][0],
        "scores": [round(1 - d, 4) for d in results["distances"][0]]
    }


def print_search_results(label: str, results: dict):
    """格式化輸出搜索結果"""
    print(f"\n{label}")
    print(f"  where 條件: {results['where_clause']}")
    print(f"  結果數量: {len(results['ids'])} 份")
    for doc_id, doc, meta, score in zip(
        results["ids"], results["documents"],
        results["metadatas"], results["scores"]
    ):
        print(f"  [{doc_id}] score={score} | cat={meta['category']}, year={meta['year']} | {doc[:50]}...")


print("✅ filtered_search 函式已定義")

In [ ]:
# ── Demo：同一查詢，有無過濾的差異 ──
test_query = "embedding 模型的比較"
print(f"查詢：'{test_query}'")
print("=" * 60)

# 1. 無過濾：搜索全部 10 份文件
r_all = filtered_search(test_query, n_results=4)
print_search_results("❶ 無過濾（全量搜索）：", r_all)

# 2. 只搜 LLM 類別
r_llm = filtered_search(test_query, category="LLM", n_results=3)
print_search_results("❷ category='LLM' 過濾：", r_llm)

# 3. 只搜 2024 年以後的文件
r_new = filtered_search(test_query, year_gte=2024, n_results=3)
print_search_results("❸ year >= 2024 過濾：", r_new)

# 4. 同時過濾：LLM 類 + 2024 年
r_llm_new = filtered_search(test_query, category="LLM", year_gte=2024, n_results=3)
print_search_results("❹ category='LLM' AND year >= 2024：", r_llm_new)

print("\n觀察：metadata 過濾讓搜索更精準，排除了不相關類別和過時文件！")

In [ ]:
# ── 示範進階 where 子句運算子 ──
print("進階 where 子句示範：")
print("=" * 60)

# $in 運算子：搜索多個 source
query_emb = create_embedding("向量資料庫如何選擇？")
r_in = meta_collection.query(
    query_embeddings=[query_emb],
    n_results=3,
    where={"source": {"$in": ["documentation", "research_paper"]}},
    include=["documents", "metadatas", "distances"]
)
print("\n$in 運算子（source in ['documentation', 'research_paper']）：")
for doc_id, meta, dist in zip(r_in["ids"][0], r_in["metadatas"][0], r_in["distances"][0]):
    print(f"  [{doc_id}] source={meta['source']}, category={meta['category']}, score={round(1-dist, 4)}")

# $or 運算子：LLM 或 production
r_or = meta_collection.query(
    query_embeddings=[query_emb],
    n_results=4,
    where={"$or": [{"category": {"$eq": "LLM"}}, {"category": {"$eq": "production"}}]},
    include=["documents", "metadatas", "distances"]
)
print("\n$or 運算子（category='LLM' OR category='production'）：")
for doc_id, meta, dist in zip(r_or["ids"][0], r_or["metadatas"][0], r_or["distances"][0]):
    print(f"  [{doc_id}] category={meta['category']}, year={meta['year']}, score={round(1-dist, 4)}")

---
## Part 3：非同步並行檢索（Async Concurrent Retrieval）

### 核心問題：多路搜索依序等待

混合搜索（Hybrid Search = 向量 + BM25）需要執行兩次搜索。如果依序執行，總時間是兩者之和：

```
Sequential（慢）:
  Vector Search → 80ms
                       BM25 Search → 5ms
                                          Total: 85ms

Concurrent（快）:
  Vector Search → 80ms ┐
  BM25 Search → 5ms    ┘ → Total: 80ms（並行！）
```

### asyncio 的關鍵概念

- `async def`：宣告非同步函式
- `await`：等待一個非同步操作完成（期間可執行其他任務）
- `asyncio.gather()`：同時啟動多個 coroutine，等全部完成
- `loop.run_in_executor()`：在 thread pool 中執行阻塞式（blocking）函式

> **Jupyter 說明：** Jupyter Notebook 本身已有 event loop，可以直接用 `await` 呼叫 async 函式，不需要 `asyncio.run()`。

In [ ]:
# ── 建立 BM25 索引（從前面的文件）──
try:
    from rank_bm25 import BM25Okapi
    import re

    def tokenize_zh(text: str) -> list:
        """簡單的中文字元級分詞"""
        return list(re.sub(r'\s+', '', text))

    bm25_corpus = [doc["content"] for doc in documents_with_metadata]
    bm25_tokenized = [tokenize_zh(doc) for doc in bm25_corpus]
    bm25_index = BM25Okapi(bm25_tokenized)
    print("✅ BM25 索引已建立")
except ImportError:
    print("⚠️  rank_bm25 未安裝，BM25 搜索將使用模擬版本")
    bm25_index = None


def vector_search_sync(query: str, n_results: int = 5) -> list:
    """同步向量搜索（封裝 ChromaDB 查詢）"""
    # 模擬搜索延遲
    time.sleep(random.uniform(0.06, 0.10))  # 60-100ms
    query_emb = create_embedding(query)
    results = meta_collection.query(
        query_embeddings=[query_emb],
        n_results=min(n_results, meta_collection.count()),
        include=["documents", "distances"]
    )
    return [
        {"id": id_, "content": doc, "score": round(1 - dist, 4), "source": "vector"}
        for id_, doc, dist in zip(
            results["ids"][0], results["documents"][0], results["distances"][0]
        )
    ]


def bm25_search_sync(query: str, n_results: int = 5) -> list:
    """同步 BM25 搜索"""
    # 模擬搜索延遲（BM25 通常比向量搜索快）
    time.sleep(random.uniform(0.003, 0.008))  # 3-8ms
    if bm25_index is None:
        return []
    tokens = tokenize_zh(query)
    scores = bm25_index.get_scores(tokens)
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:n_results]
    return [
        {"id": documents_with_metadata[i]["id"],
         "content": documents_with_metadata[i]["content"],
         "score": round(float(scores[i]), 4),
         "source": "bm25"}
        for i in top_idx if scores[i] > 0
    ]


def rrf_fuse(vec_results: list, bm25_results: list, k: int = 60) -> list:
    """Reciprocal Rank Fusion（NB03 學過）"""
    scores = {}
    for rank, r in enumerate(vec_results):
        scores[r["id"]] = scores.get(r["id"], 0) + 1 / (k + rank + 1)
    for rank, r in enumerate(bm25_results):
        scores[r["id"]] = scores.get(r["id"], 0) + 1 / (k + rank + 1)
    all_docs = {r["id"]: r for r in vec_results + bm25_results}
    return [
        {**all_docs[doc_id], "rrf_score": round(s, 6)}
        for doc_id, s in sorted(scores.items(), key=lambda x: x[1], reverse=True)
    ]


print("✅ 同步搜索函式已定義")

In [ ]:
async def async_vector_search(query: str, n_results: int = 5) -> list:
    """非同步向量搜索：將阻塞式呼叫丟到 thread pool 執行"""
    loop = asyncio.get_event_loop()
    result = await loop.run_in_executor(
        None,  # 使用預設 thread pool
        vector_search_sync, query, n_results
    )
    return result


async def async_bm25_search(query: str, n_results: int = 5) -> list:
    """非同步 BM25 搜索"""
    loop = asyncio.get_event_loop()
    result = await loop.run_in_executor(
        None,
        bm25_search_sync, query, n_results
    )
    return result


async def async_hybrid_search(query: str, n_results: int = 5) -> tuple:
    """並行執行向量搜索 + BM25，回傳 (fused_results, elapsed_ms)"""
    start = time.time()

    # asyncio.gather：同時啟動兩個搜索任務，等全部完成
    vec_results, bm25_results = await asyncio.gather(
        async_vector_search(query, n_results),
        async_bm25_search(query, n_results)
    )

    elapsed = (time.time() - start) * 1000  # 轉換為 ms
    fused = rrf_fuse(vec_results, bm25_results)
    return fused, elapsed


print("✅ 非同步搜索函式已定義")

In [ ]:
# ── 效能基準測試：Sequential vs Concurrent ──
benchmark_query = "向量資料庫如何處理高維度向量？"

print(f"基準查詢：'{benchmark_query}'")
print("=" * 55)

# 執行多輪取平均，降低隨機性
n_rounds = 5
seq_times = []
conc_times = []

for i in range(n_rounds):
    # Sequential（依序執行）
    start = time.time()
    r1 = vector_search_sync(benchmark_query, 5)
    r2 = bm25_search_sync(benchmark_query, 5)
    _ = rrf_fuse(r1, r2)
    seq_ms = (time.time() - start) * 1000
    seq_times.append(seq_ms)

    # Concurrent（並行執行）
    fused, conc_ms = await async_hybrid_search(benchmark_query, 5)
    conc_times.append(conc_ms)

avg_seq = sum(seq_times) / n_rounds
avg_conc = sum(conc_times) / n_rounds
speedup = avg_seq / avg_conc

print(f"\n{'方法':<15} {'平均時間':>10} {'最小':>8} {'最大':>8}")
print("-" * 45)
print(f"{'Sequential':<15} {avg_seq:>9.1f}ms {min(seq_times):>7.1f}ms {max(seq_times):>7.1f}ms")
print(f"{'Concurrent':<15} {avg_conc:>9.1f}ms {min(conc_times):>7.1f}ms {max(conc_times):>7.1f}ms")
print(f"\n加速比: {speedup:.2f}x（{n_rounds} 輪平均）")

print("\n並行搜索結果（RRF 融合後 Top-3）：")
for r in fused[:3]:
    print(f"  [{r['id']}] rrf={r['rrf_score']} | {r['content'][:55]}...")

---
## Part 4：成本優化策略（Cost Optimization）

### Token 費用結構

每次 RAG 查詢的費用由三部分組成：

```
使用者查詢
  ↓
Embedding API 費用（把 query 轉向量）
  ↓
LLM Input 費用（system prompt + context chunks + query）
  ↓
LLM Output 費用（生成的答案）
```

### 2024 年 OpenAI 定價（每 1M tokens）

| 模型 | Input | Output | 相對成本 |
|------|-------|--------|----------|
| `gpt-4o` | $5.00 | $15.00 | 基準 |
| `gpt-4o-mini` | $0.15 | $0.60 | ~30x 便宜 |
| `text-embedding-3-small` | $0.02 | — | 極低 |
| `text-embedding-3-large` | $0.13 | — | 嵌入中等 |

In [ ]:
def estimate_rag_cost(query: str, contexts: list, answer: str,
                      model: str = "gpt-4o-mini") -> dict:
    """估算單次 RAG 查詢的 API 成本"""
    enc = tiktoken.get_encoding("cl100k_base")

    # ── Token 計算 ──
    query_tokens = len(enc.encode(query))
    context_tokens = sum(len(enc.encode(c)) for c in contexts)
    system_overhead = 200  # system prompt 固定開銷
    llm_input_tokens = query_tokens + context_tokens + system_overhead
    llm_output_tokens = len(enc.encode(answer))

    # ── 定價（per 1M tokens，2024 年）──
    pricing = {
        "gpt-4o-mini":              {"input": 0.15,  "output": 0.60},
        "gpt-4o":                   {"input": 5.00,  "output": 15.00},
        "text-embedding-3-small":   {"input": 0.02,  "output": 0.0},
        "text-embedding-3-large":   {"input": 0.13,  "output": 0.0},
    }

    emb_cost = query_tokens * pricing["text-embedding-3-small"]["input"] / 1_000_000
    llm_cost = (
        llm_input_tokens  * pricing[model]["input"] +
        llm_output_tokens * pricing[model]["output"]
    ) / 1_000_000

    return {
        "model":               model,
        "query_tokens":        query_tokens,
        "context_tokens":      context_tokens,
        "output_tokens":       llm_output_tokens,
        "total_input_tokens":  llm_input_tokens,
        "embedding_cost_usd":  round(emb_cost, 8),
        "llm_cost_usd":        round(llm_cost, 8),
        "total_cost_usd":      round(emb_cost + llm_cost, 8),
    }


# ── 示範：計算一次 RAG 查詢的成本 ──
sample_query = "RAG 系統如何優化延遲？"
sample_contexts = [
    documents_with_metadata[4]["content"],  # d05
    documents_with_metadata[5]["content"],  # d06
    documents_with_metadata[8]["content"],  # d09
]
sample_answer = "RAG 系統的延遲優化主要有幾個方向：首先是語意快取，對重複查詢直接回傳快取答案。其次是非同步並行搜索，同時執行向量搜索和 BM25。第三是使用更小的模型（如 gpt-4o-mini）處理簡單查詢。"

print("單次 RAG 查詢成本估算：")
print("=" * 50)

for model_name in ["gpt-4o-mini", "gpt-4o"]:
    cost = estimate_rag_cost(sample_query, sample_contexts, sample_answer, model_name)
    print(f"\n模型: {model_name}")
    print(f"  Query tokens:   {cost['query_tokens']}")
    print(f"  Context tokens: {cost['context_tokens']}")
    print(f"  Output tokens:  {cost['output_tokens']}")
    print(f"  Embedding cost: ${cost['embedding_cost_usd']:.8f}")
    print(f"  LLM cost:       ${cost['llm_cost_usd']:.8f}")
    print(f"  Total cost:     ${cost['total_cost_usd']:.8f}")

In [ ]:
# ── 規模化成本試算（每天 10,000 次查詢）──
daily_queries = 10_000
cache_hit_rate = 0.40  # 語意快取命中率 40%
context_compression_ratio = 0.60  # contextual compression 節省 40%

cost_before = estimate_rag_cost(sample_query, sample_contexts, sample_answer, "gpt-4o-mini")

# 優化前
daily_cost_before = cost_before["total_cost_usd"] * daily_queries

# 優化後（快取 + 壓縮）
effective_queries = daily_queries * (1 - cache_hit_rate)  # 快取過濾後的有效查詢數
compressed_contexts = [
    c[:int(len(c) * context_compression_ratio)] for c in sample_contexts
]
cost_after = estimate_rag_cost(sample_query, compressed_contexts, sample_answer, "gpt-4o-mini")
daily_cost_after = cost_after["total_cost_usd"] * effective_queries

saving_pct = (1 - daily_cost_after / daily_cost_before) * 100

print("規模化成本估算（每天 10,000 次查詢）")
print("=" * 55)
print(f"\n{'項目':<30} {'優化前':>12} {'優化後':>12}")
print("-" * 55)
print(f"{'每次 context tokens':<30} {cost_before['context_tokens']:>12} {cost_after['context_tokens']:>12}")
print(f"{'有效查詢數/天':<30} {daily_queries:>12,} {int(effective_queries):>12,}")
print(f"{'每次查詢成本 (USD)':<30} ${cost_before['total_cost_usd']:>11.6f} ${cost_after['total_cost_usd']:>11.6f}")
print(f"{'每日總成本 (USD)':<30} ${daily_cost_before:>11.4f} ${daily_cost_after:>11.4f}")
print(f"{'每月總成本 (USD)':<30} ${daily_cost_before*30:>11.2f} ${daily_cost_after*30:>11.2f}")
print("-" * 55)
print(f"\n成本節省: {saving_pct:.1f}%")
print(f"\n優化手段:")
print(f"  ✅ 語意快取（命中率 {cache_hit_rate*100:.0f}%）→ 節省 {cache_hit_rate*100:.0f}% LLM 呼叫")
print(f"  ✅ Contextual Compression → context tokens 減少 {(1-context_compression_ratio)*100:.0f}%")
print(f"  ✅ 使用 gpt-4o-mini（而非 gpt-4o，相差 ~30x）")

In [ ]:
# ── 成本比較長條圖 ──
fig, ax = plt.subplots(figsize=(11, 5))

scenarios = [
    "gpt-4o\n無優化",
    "gpt-4o-mini\n無優化",
    "gpt-4o-mini\n+語意快取",
    "gpt-4o-mini\n+快取+壓縮"
]

# 計算各情境的每日成本
cost_4o = estimate_rag_cost(sample_query, sample_contexts, sample_answer, "gpt-4o")
cost_mini = estimate_rag_cost(sample_query, sample_contexts, sample_answer, "gpt-4o-mini")
cost_mini_cache = cost_mini["total_cost_usd"] * daily_queries * (1 - cache_hit_rate)
cost_mini_both = cost_after["total_cost_usd"] * effective_queries

daily_costs = [
    cost_4o["total_cost_usd"] * daily_queries,
    cost_mini["total_cost_usd"] * daily_queries,
    cost_mini_cache,
    cost_mini_both
]

colors = ["#EF5350", "#FF9800", "#42A5F5", "#4CAF50"]
bars = ax.bar(scenarios, daily_costs, color=colors, width=0.5,
               edgecolor="white", linewidth=1.5)

for bar, cost in zip(bars, daily_costs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(daily_costs) * 0.01,
            f"${cost:.2f}", ha="center", va="bottom", fontsize=11, fontweight="bold")

# 添加節省百分比標籤
for i in range(1, len(daily_costs)):
    pct = (1 - daily_costs[i] / daily_costs[0]) * 100
    ax.text(i, daily_costs[i] / 2, f"-{pct:.0f}%",
            ha="center", va="center", fontsize=10, color="white", fontweight="bold")

ax.set_title(f"每日成本比較（{daily_queries:,} 次查詢/天）", fontsize=14)
ax.set_ylabel("每日費用 (USD)")
ax.grid(axis="y", alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()
print("圖表已生成")

---
## Part 5：Self-RAG（自主檢索決策）

### 什麼是 Agentic RAG？

傳統 RAG 是「被動的」：每次查詢都強制執行完整的檢索-生成流程，不管問題是否需要外部知識。

**Agentic RAG** 讓 LLM 成為主動的 **Agent**，自主決定：
- 是否需要檢索？
- 檢索到的文件是否相關？
- 生成的答案是否有幻覺？

### Self-RAG 的流程

```
                    ┌─ 不需要檢索 → 直接生成
查詢 → 需要檢索？ ─┤
                    └─ 需要檢索 → 檢索 → 文件相關嗎？
                                           ├─ 相關 → 生成 → 有幻覺嗎？
                                           │               ├─ 無 → 輸出
                                           │               └─ 有 → 重新生成
                                           └─ 不相關 → 修正查詢重試
```

### 三個關鍵的 Special Token 判斷（論文中的設計）

| Token | 判斷內容 | 選項 |
|-------|----------|------|
| `[Retrieve]` | 是否需要檢索？ | YES / NO |
| `[IsRel]` | 文件是否相關？ | RELEVANT / IRRELEVANT |
| `[IsHall]` | 答案是否有幻覺？ | SUPPORTED / HALLUCINATED |

In [ ]:
def judge_retrieval_need(query: str) -> str:
    """判斷問題是否需要外部知識（[Retrieve] token 的模擬）"""
    prompt = f"""判斷回答以下問題是否需要查詢外部知識庫。

問題：{query}

規則：
- 需要特定事實、最新資訊、或專業領域知識 → 回答 YES
- 常識性問題、數學/邏輯問題、不需要特定知識 → 回答 NO

只回答 YES 或 NO，不要有任何其他文字。"""

    result = llm_call(prompt, temperature=0)
    return "YES" if "YES" in result.upper() else "NO"


def judge_relevance(query: str, doc_content: str) -> str:
    """判斷文件是否和查詢相關（[IsRel] token 的模擬）"""
    prompt = f"""判斷以下文件是否和問題相關，能幫助回答該問題。

問題：{query}

文件：{doc_content[:300]}

只回答 RELEVANT 或 IRRELEVANT。"""

    result = llm_call(prompt, temperature=0)
    return "RELEVANT" if "RELEVANT" in result.upper() else "IRRELEVANT"


def check_hallucination(answer: str, docs: list) -> str:
    """檢查生成答案是否有幻覺（[IsHall] token 的模擬）"""
    docs_text = "\n".join([d.get("content", d) if isinstance(d, dict) else d for d in docs[:3]])
    prompt = f"""判斷以下答案中的關鍵事實是否可以在給定文件中找到支撐。

文件：
{docs_text[:600]}

答案：{answer[:400]}

如果答案的主要事實都有文件支撐，回答 SUPPORTED。
如果答案包含文件中沒有的捏造資訊，回答 HALLUCINATED。
只回答 SUPPORTED 或 HALLUCINATED。"""

    result = llm_call(prompt, temperature=0)
    return "SUPPORTED" if "SUPPORTED" in result.upper() else "HALLUCINATED"


def generate_with_context(query: str, docs: list) -> str:
    """根據檢索文件生成回答"""
    context = "\n\n".join([
        d["content"] if isinstance(d, dict) else d for d in docs
    ])
    return llm_call(
        f"參考資料：\n{context}\n\n問題：{query}\n\n請根據參考資料回答，用繁體中文。",
        temperature=0.1
    )


def direct_answer(query: str) -> str:
    """不需要外部知識時，直接由 LLM 回答"""
    return llm_call(f"請回答：{query}", temperature=0.1)


print("✅ Self-RAG 輔助函式已定義")

In [ ]:
def self_rag(query: str, max_retries: int = 2, verbose: bool = True) -> dict:
    """
    Self-RAG：模型自主決定是否需要檢索、評估文件相關性、檢查輸出幻覺
    """
    if verbose:
        print(f"\n[Self-RAG] 查詢: {query}")
        print("-" * 50)

    # ── Step 1：判斷是否需要檢索 ──
    retrieval_need = judge_retrieval_need(query)
    if verbose:
        print(f"  [Retrieve] 是否需要檢索？ → {retrieval_need}")

    if retrieval_need == "NO":
        answer = direct_answer(query)
        return {"query": query, "answer": answer, "retrieved": False,
                "path": "direct", "relevant_docs_count": 0}

    # ── Step 2：向量搜索 ──
    query_emb = create_embedding(query)
    raw_results = meta_collection.query(
        query_embeddings=[query_emb],
        n_results=5,
        include=["documents"]
    )
    candidates = [
        {"id": id_, "content": doc}
        for id_, doc in zip(raw_results["ids"][0], raw_results["documents"][0])
    ]

    # ── Step 3：評估每個文件的相關性 ──
    relevant_docs = []
    for doc in candidates:
        relevance = judge_relevance(query, doc["content"])
        if verbose:
            print(f"  [IsRel] [{doc['id']}] → {relevance}")
        if relevance == "RELEVANT":
            relevant_docs.append(doc)

    if not relevant_docs:
        if verbose:
            print("  → 沒有相關文件，觸發 CRAG fallback")
        return {"query": query,
                "answer": "根據現有知識庫資料，無法準確回答此問題。",
                "retrieved": True, "path": "crag_fallback", "relevant_docs_count": 0}

    # ── Step 4：生成答案 ──
    answer = generate_with_context(query, relevant_docs)

    # ── Step 5：幻覺檢查 ──
    hall_check = check_hallucination(answer, relevant_docs)
    if verbose:
        print(f"  [IsHall] 幻覺檢查 → {hall_check}")

    if hall_check == "HALLUCINATED" and max_retries > 0:
        if verbose:
            print(f"  → 發現幻覺，重試（剩餘 {max_retries-1} 次）")
        return self_rag(query, max_retries - 1, verbose)

    return {
        "query": query,
        "answer": answer,
        "retrieved": True,
        "path": "full_rag",
        "relevant_docs_count": len(relevant_docs),
        "hallucination_check": hall_check
    }


print("✅ self_rag 函式已定義")

In [ ]:
# ── Demo：三種不同類型的查詢 ──
test_cases = [
    {"query": "ChromaDB 支援哪兩種部署模式？",
     "desc": "需要檢索（知識庫有答案）"},
    {"query": "1 + 1 等於幾？",
     "desc": "不需要檢索（數學問題）"},
    {"query": "2026 年最新發布的 LLM 模型有哪些？",
     "desc": "需要檢索（知識庫無此資訊，CRAG fallback）"},
]

print("Self-RAG 三種情境示範")
print("=" * 60)

self_rag_results = []
for tc in test_cases:
    print(f"\n情境：{tc['desc']}")
    result = self_rag(tc["query"], verbose=True)
    self_rag_results.append(result)
    print(f"\n  最終路徑: {result['path']}")
    print(f"  答案: {result['answer'][:150]}..." if len(result['answer']) > 150 else f"  答案: {result['answer']}")
    print()

print("\n路徑統計：")
paths = [r["path"] for r in self_rag_results]
for path in set(paths):
    print(f"  {path}: {paths.count(path)} 次")

---
## Part 6：CRAG（Corrective RAG，修正式檢索）

### 核心概念：檢索品質評估 + 自動修正

Self-RAG 決定「要不要檢索」，而 CRAG 解決「**檢索到的東西夠不夠好**」的問題。

```
檢索 → 評估文件品質
  ├── CORRECT（相關度高）: 直接生成
  ├── INCORRECT（不相關）: 觸發修正
  │   ├── 重新改寫查詢（Query Rewriting）
  │   ├── 擴展搜索範圍
  │   └── 若仍不足 → 承認無法回答
  └── AMBIGUOUS（模糊）: 結合多種策略
```

### CRAG vs Self-RAG 的差別

| 面向 | Self-RAG | CRAG |
|------|----------|------|
| **決策點** | 檢索前（要不要撈？） | 檢索後（撈得好不好？） |
| **幻覺處理** | 生成後檢查幻覺 | 確保 context 品質，從源頭防幻覺 |
| **修正機制** | 重試 | 改寫查詢 + 擴展搜索 |
| **適用場景** | 混合知識需求（有些問不需要 RAG） | 所有問題都需要 RAG，但擔心品質 |

In [ ]:
def evaluate_retrieval_quality(query: str, docs: list) -> str:
    """評估檢索結果的整體品質：CORRECT / INCORRECT / AMBIGUOUS"""
    docs_text = "\n".join([
        f"[{i+1}] {d['content'][:150]}"
        for i, d in enumerate(docs[:3])
    ])
    prompt = f"""評估以下檢索到的文件對於回答問題的整體品質。

問題：{query}

檢索文件：
{docs_text}

評估標準：
- CORRECT：文件高度相關，可以直接回答問題
- INCORRECT：文件不相關或完全無法回答問題
- AMBIGUOUS：文件部分相關，但資訊不完整

只回答 CORRECT、INCORRECT 或 AMBIGUOUS。"""

    result = llm_call(prompt, temperature=0)
    for label in ["CORRECT", "INCORRECT", "AMBIGUOUS"]:
        if label in result.upper():
            return label
    return "AMBIGUOUS"


def rewrite_query(query: str) -> str:
    """重新改寫查詢，使其更容易找到相關文件"""
    prompt = f"""請將以下問題改寫成更容易在知識庫中搜索到答案的形式。
使用更多專業術語和關鍵字，讓搜索更精準。只輸出改寫後的問題，不要解釋。

原始問題：{query}"""
    return llm_call(prompt, temperature=0.3)


def deduplicate(docs: list) -> list:
    """根據 id 去重"""
    seen = set()
    result = []
    for doc in docs:
        doc_id = doc.get("id", doc.get("content", "")[:30])
        if doc_id not in seen:
            seen.add(doc_id)
            result.append(doc)
    return result


def vector_search_for_crag(query: str, n_results: int = 3) -> list:
    """CRAG 用的向量搜索包裝"""
    query_emb = create_embedding(query)
    raw = meta_collection.query(
        query_embeddings=[query_emb],
        n_results=min(n_results, meta_collection.count()),
        include=["documents"]
    )
    return [
        {"id": id_, "content": doc}
        for id_, doc in zip(raw["ids"][0], raw["documents"][0])
    ]


print("✅ CRAG 輔助函式已定義")

In [ ]:
def crag_retrieve_and_generate(query: str, verbose: bool = True) -> dict:
    """Corrective RAG：檢索品質評估 + 自動修正策略"""
    if verbose:
        print(f"\n[CRAG] 查詢: {query}")
        print("-" * 50)

    # ── Step 1：初始檢索 ──
    results = vector_search_for_crag(query, n_results=3)
    if verbose:
        print(f"  [初始檢索] 找到 {len(results)} 份文件: {[r['id'] for r in results]}")

    # ── Step 2：評估品質 ──
    quality = evaluate_retrieval_quality(query, results)
    if verbose:
        print(f"  [品質評估] → {quality}")

    if quality == "CORRECT":
        # 高品質：直接生成
        answer = generate_with_context(query, results)
        return {"query": query, "answer": answer, "quality": quality,
                "action": "direct", "final_docs": [r["id"] for r in results]}

    elif quality == "INCORRECT":
        # 低品質：嘗試改寫查詢
        rewritten = rewrite_query(query)
        if verbose:
            print(f"  [查詢改寫] → {rewritten}")

        new_results = vector_search_for_crag(rewritten, n_results=5)
        new_quality = evaluate_retrieval_quality(query, new_results)
        if verbose:
            print(f"  [改寫後品質] → {new_quality}")

        if new_quality == "INCORRECT":
            # 仍然不夠好：承認無法回答
            return {"query": query,
                    "answer": "我在知識庫中找不到足夠的資訊來準確回答這個問題。",
                    "quality": quality, "action": "rejected",
                    "rewritten_query": rewritten, "final_docs": []}

        answer = generate_with_context(query, new_results)
        return {"query": query, "answer": answer, "quality": quality,
                "action": "corrected", "rewritten_query": rewritten,
                "final_docs": [r["id"] for r in new_results]}

    else:  # AMBIGUOUS
        # 模糊：擴展搜索，合併兩次結果
        rewritten = rewrite_query(query)
        extended_results = vector_search_for_crag(rewritten, n_results=3)
        combined = deduplicate(results + extended_results)[:5]
        if verbose:
            print(f"  [擴展搜索] 合併後共 {len(combined)} 份文件")
        answer = generate_with_context(query, combined)
        return {"query": query, "answer": answer, "quality": quality,
                "action": "augmented",
                "final_docs": [r["id"] for r in combined]}


print("✅ crag_retrieve_and_generate 函式已定義")

In [ ]:
# ── Demo：三種品質情境 ──
crag_cases = [
    {"query": "語意快取如何節省 LLM 費用？",
     "desc": "CORRECT 情境（知識庫有直接相關文件）"},
    {"query": "Python 的 async/await 語法如何使用？",
     "desc": "INCORRECT 情境（知識庫無程式語法教學）"},
    {"query": "向量資料庫在生產環境的部署考量？",
     "desc": "AMBIGUOUS 情境（有部分相關但不完整）"},
]

print("CRAG 三種情境示範")
print("=" * 60)

for tc in crag_cases:
    print(f"\n情境：{tc['desc']}")
    r = crag_retrieve_and_generate(tc["query"], verbose=True)
    print(f"\n  執行動作: {r['action']}")
    if 'rewritten_query' in r:
        print(f"  改寫後查詢: {r['rewritten_query']}")
    print(f"  使用文件: {r.get('final_docs', [])}")
    ans = r['answer']
    print(f"  答案: {ans[:120]}..." if len(ans) > 120 else f"  答案: {ans}")
    print()

---
## Part 7：完整生產級 RAG 架構圖

整合本系列七個 Notebook 所學的所有技術，展示一個完整的生產級 RAG 系統架構：

```
                    生產級 RAG 系統架構

═══════════════════ 離線索引流程 ═══════════════════

原始文件
  ↓
文件解析（PDF / HTML / DB）                          [NB02]
  ↓
Parent-Child Chunking（語義切塊）                    [NB02]
  ↓
Embedding（text-embedding-3-small）                  [NB01]
  ↓
ChromaDB 向量儲存 + Metadata 儲存                    [NB01, NB07 Part2]

════════════════════ 線上查詢流程 ════════════════════

使用者查詢
  ↓
[NB07 Part1]  語意快取查詢 ──→ Cache Hit → 直接回傳（省 LLM 費用）
  ↓ (Cache Miss)
[NB07 Part5]  Self-RAG：需要檢索？
  ├── NO → 直接 LLM 回答
  └── YES ↓
[NB05]        Query Transformation（HyDE / Multi-Query / Step-Back）
  ↓
[NB07 Part2]  Metadata 過濾（縮小搜索範圍）
  ↓
[NB07 Part3]  Async 混合搜索（向量搜索 + BM25，並行執行）
  ↓
[NB03]        Reranking（Cross-Encoder 重排序）
  ↓
[NB07 Part6]  CRAG 品質評估
  ├── CORRECT → 繼續
  ├── INCORRECT → 改寫查詢重試
  └── AMBIGUOUS → 擴展搜索
  ↓
[NB06]        Contextual Compression（壓縮 context，省 token）
  ↓
[NB06]        Context 排序（Lost-in-Middle 防範）
  ↓
LLM 生成（gpt-4o-mini）
  ↓
[NB07 Part1]  存入語意快取
  ↓
回傳使用者
  ↓
[NB04]        RAGAS 評估（Context Recall / Faithfulness / Answer Relevancy）
```

### 各技術的效益總結

| 技術 | 學習 NB | 主要效益 | 典型提升 |
|------|---------|----------|----------|
| 語意快取 | NB07 | 節省 LLM 費用 | 30-60% 成本↓ |
| Metadata 過濾 | NB07 | 提升精準度 | 精確度↑，搜索範圍↓ |
| 非同步並行 | NB07 | 降低延遲 | 10-30% 延遲↓ |
| Self-RAG | NB07 | 智慧決策 | 避免不必要的 RAG |
| CRAG | NB07 | 修正壞結果 | Hallucination↓ |
| Query Transform | NB05 | 提升召回率 | Recall↑ 20-40% |
| Hybrid Search | NB03 | 更全面的檢索 | Recall↑ 15-25% |
| Reranking | NB03 | 提升精確度 | Precision↑ 10-20% |
| Parent-Child Chunking | NB02 | 保留上下文 | Answer Quality↑ |
| RAGAS 評估 | NB04 | 量化品質 | 可持續改善 |

---
## 面試關鍵問答（FDE Interview Q&A）

---

**面試官：** RAG 系統在生產環境中最常見的成本問題是什麼？

**好答案：** 「生產 RAG 的成本問題主要有三層：第一是重複查詢——使用者常問相似問題，每次都呼叫 LLM 非常浪費，解法是語意快取（Semantic Cache），用向量相似度識別近似查詢，命中率通常可達 30-60%，每天可省數百美元。第二是 context 過長——把太多 chunk 塞進 prompt 導致 input token 暴增，解法是 contextual compression 和 reranking，只保留最相關的部分，可減少 50-70% context tokens。第三是模型選擇——很多查詢用 gpt-4o-mini 就夠了，比 gpt-4o 便宜 30 倍，透過查詢分類路由到合適的模型是最快的降本手段。」

---

**面試官：** 如何設計一個高可用（High Availability）的 RAG 系統？

**好答案：** 「高可用 RAG 系統有幾個關鍵設計。首先是快取層：語意快取讓大部分流量不需要完整 pipeline，即使後端壓力大也能快速回應。其次是降級策略（Graceful Degradation）：當 reranker 超時就跳過、當 LLM 延遲太高就用更小的模型。第三是非同步架構：混合搜索用 asyncio 並行，向量搜索和 BM25 同時跑不互相等待。第四是 Circuit Breaker：當某個依賴服務（如 OpenAI API）連續失敗，自動切換到備用路徑或回傳快取。最後是監控：持續追蹤 P50/P95 延遲、快取命中率、RAGAS 評分，超過閾值就告警。」

---

**面試官：** Self-RAG 和普通 RAG 的核心區別是什麼？實際系統中值得用嗎？

**好答案：** 「核心區別是『主動性』。普通 RAG 是被動的管道——每次查詢都強制執行『檢索→生成』，不管問題是否需要外部知識。Self-RAG 讓 LLM 在三個關鍵點做決策：一、[Retrieve] 要不要檢索？二、[IsRel] 每份文件相關嗎？三、[IsHall] 生成的答案有沒有幻覺？

實際上值不值得用取決於場景。如果你的系統有大量『常識問題』（如計算、定義），每次都觸發 RAG 是浪費——Self-RAG 可以節省 20-30% 的無效檢索。但 Self-RAG 每個判斷都需要 LLM 呼叫，如果問題本來就全部需要檢索，反而增加延遲和成本。最實用的做法是輕量規則過濾（如正則判斷是否為數學題）搭配簡單的 [Retrieve] 判斷，不必每次都問 LLM。」

---

**面試官：** 什麼情況下 RAG 無法解決問題，需要考慮其他方案？

**好答案：** 「RAG 有幾個結構性限制。第一是需要特定輸出格式的場景——RAG 難以讓 LLM 穩定輸出特定的 JSON 結構或語氣風格，這時候 Fine-tuning 更合適。第二是知識庫無法覆蓋的推理任務——如果需要複雜數學推理或多步邏輯推導，靠 RAG 拿回 context 沒有幫助，應考慮 Chain-of-Thought 或 Tool-use。第三是極高精準度要求的領域——醫療、法律等場景，文件有輕微語義偏差就可能導致危險答案，這時候需要更嚴格的 Agentic 驗證循環或人工審核。第四是知識高度碎片化的場景——當一個問題的答案分散在 100 個文件的微小細節中，RAG 難以有效聚合，Graph RAG（知識圖譜增強）是更好的方向。」

---

**面試官：** CRAG 和 Self-RAG 可以同時使用嗎？

**好答案：** 「可以，而且這正是生產系統常見的設計。兩者解決不同環節的問題：Self-RAG 的 [Retrieve] 決策在**前端**——省掉不必要的 RAG；CRAG 的品質評估在**後端**——確保真的要 RAG 時品質夠好。組合使用的流程：先用 Self-RAG 判斷是否需要檢索，若需要才進入 CRAG 的品質評估循環。這樣既避免了無謂的檢索，又在確實需要 RAG 時保證了檢索品質。代價是每次查詢的 LLM 呼叫次數增加（[Retrieve] + 品質評估），需要在品質和延遲之間取捨，可以透過語意快取緩解這個問題。」

---

---
## 整個系列總回顧

### NB01~NB07 系列學習地圖

| NB | 主題 | 核心技術 | 關鍵收穫 |
|----|------|----------|----------|
| **NB01** | RAG 基礎與完整流程 | ChromaDB + OpenAI Embedding | 理解 RAG 五步驟、向量相似度原理 |
| **NB02** | Chunking 文字切塊策略 | Fixed / Recursive / Parent-Child | Chunk size 對召回品質的影響 |
| **NB03** | 混合搜索與重排序 | BM25 + RRF + Cross-Encoder | Hybrid Search 大幅提升 Recall |
| **NB04** | RAG 評估框架 | RAGAS (Faithfulness, Recall, Relevancy) | 量化 RAG 品質的三大指標 |
| **NB05** | Query 轉換技術 | HyDE / Multi-Query / Step-Back / Decomposition | 改善「垃圾進垃圾出」問題 |
| **NB06** | 進階 RAG 技術 | Contextual Compression / Lost-in-Middle | 優化 context 品質和排序 |
| **NB07** | 生產優化與 Agentic RAG | Semantic Cache / Metadata Filter / Self-RAG / CRAG | 從 Demo 到生產的工程現實 |

### RAG 技術演進路線

```
Naive RAG          → Advanced RAG        → Modular RAG       → Agentic RAG
────────────          ─────────────          ────────────        ─────────────
固定 pipeline         優化各環節              模組化組合           自主決策
NB01 基礎             NB02-05 優化            NB06 壓縮/排序       NB07 Self-RAG
                                                                   CRAG
                                                                   Semantic Cache
```

### 恭喜完成 RAG 學習系列！

完成本系列後，你已具備：
- **理論基礎**：理解 RAG 每個環節的原理和取捨
- **實作能力**：用 ChromaDB + OpenAI 從零建立完整 RAG 系統
- **優化思維**：知道成本、延遲、品質的權衡點
- **面試準備**：能深入回答 Google FDE 面試中的 RAG 相關問題

### 下一步學習方向

| 方向 | 技術 | 說明 |
|------|------|------|
| **Graph RAG** | Neo4j + LangGraph | 知識圖譜增強，處理複雜關係 |
| **Multi-modal RAG** | GPT-4V / Gemini | 圖片、表格的理解和檢索 |
| **Long-context RAG** | Gemini 1M context | 極長文件的處理策略 |
| **RAG Agent** | LangGraph / CrewAI | 多步驟自主 RAG Agent |
| **Streaming RAG** | SSE / WebSocket | 即時串流輸出提升 UX |

In [ ]:
# ── 本 Notebook 最終統計 ──
print("=" * 55)
print("NB07 完成！學習成果總覽")
print("=" * 55)

achievements = [
    ("語意快取",         f"命中率 {cache.hit_rate*100:.0f}%，節省 {(len(simulated_queries) - llm_call_count)} 次 LLM 呼叫"),
    ("Metadata 過濾",    "示範 $eq / $in / $or / $and / $gte 五種運算子"),
    ("非同步並行",       f"平均加速 {speedup:.2f}x（asyncio.gather）"),
    ("成本優化",         f"透過快取+壓縮節省 {saving_pct:.0f}% 每日費用"),
    ("Self-RAG",         "三路徑：direct / full_rag / crag_fallback"),
    ("CRAG",             "三動作：direct / corrected / augmented / rejected"),
    ("生產架構圖",       "整合 NB01-NB07 所有技術的完整架構"),
]

for title, detail in achievements:
    print(f"  ✅ {title:<12} → {detail}")

print()
print("整個 RAG 系列（NB01-NB07）已全部完成！")
print("準備好迎接 Google FDE 面試了！")